# 04 · Validation selection → locked test → robustness → gates (A100)
**Rule:** choose checkpoint + thresholds on val only. Write the chosen run name into `reports/preregistration.json` *before* any `--split test` call.
Runtime → Change runtime type → **A100 GPU** (L4 also works). Note: `latency_p95_s` measured here is server-GPU latency, not on-device; compare systems only when run on the same GPU type.

## 0. Session setup (run every session)
Code is cloned from GitHub (`moghbaie/InspectEdge_Mobile_VLM`); nothing needs to be uploaded.
Google Drive (`MyDrive/InspectEdge`) only stores outputs: reports, manifests, checkpoints.
Runtime → Change runtime type → **GPU** (T4 is enough for 500M; L4/A100 for 3B).

In [ ]:
import os, json, subprocess
from google.colab import drive
drive.mount('/content/drive')
REPO  = 'https://github.com/moghbaie/InspectEdge_Mobile_VLM.git'
DRIVE = '/content/drive/MyDrive/InspectEdge'   # persistent outputs: reports, manifests, checkpoints
WORK  = '/content/InspectEdge'                 # fast local disk: code + raw images
os.makedirs(DRIVE, exist_ok=True)
if os.path.isdir(f'{WORK}/.git'):
    !git -C {WORK} pull -q --ff-only
else:
    !git clone -q {REPO} {WORK}
assert os.path.exists(f'{WORK}/pyproject.toml'), 'clone failed: check REPO URL / repo visibility'
%cd {WORK}
!git log --oneline -1
!pip -q install -r requirements-colab.txt
!pip -q install -e . --no-deps
os.environ['INSPECTEDGE_ROOT'] = WORK
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch, transformers, peft
assert transformers.__version__ == '4.57.1', 'pinned transformers not active: Runtime -> Restart session, then re-run this cell'
print('torch', torch.__version__, 'cuda', torch.version.cuda, '| transformers', transformers.__version__, '| peft', peft.__version__)
assert torch.cuda.is_available(), 'No GPU: Runtime -> Change runtime type -> GPU'

In [ ]:
# Data: official VisA release (CC BY 4.0). Verified by SHA-256 before use.
VISA_SHA = '2eb8690c803ab37de0324772964100169ec8ba1fa3f7e94291c9ca673f40f362'
import pandas as pd
def visa_missing():   # every manifest image/mask must be on local disk (a runtime reset or partial extract loses them)
    man = pd.read_json('data/manifests/visa_manifest.jsonl', lines=True)
    return [p for p in list(man.image_path) + [m for m in man.mask_path if m] if not os.path.exists(p)]
if visa_missing():
    !rm -rf data/raw/visa data/raw/_dl
    os.makedirs('data/raw/_dl', exist_ok=True); os.makedirs('data/raw/visa', exist_ok=True)
    !wget -q -O data/raw/_dl/VisA_20220922.tar https://amazon-visual-anomaly.s3.us-west-2.amazonaws.com/VisA_20220922.tar
    h = subprocess.run(['sha256sum', 'data/raw/_dl/VisA_20220922.tar'], capture_output=True, text=True).stdout.split()[0]
    assert h == VISA_SHA, f'VisA archive checksum mismatch: {h}'
    !tar -xf data/raw/_dl/VisA_20220922.tar -C data/raw/visa && rm data/raw/_dl/VisA_20220922.tar
_miss = visa_missing()
assert not _miss, f'{len(_miss)} VisA files missing after extraction (disk full?), e.g. {_miss[0]}'
open('data/raw/visa/.complete', 'w').close()
!ls data/raw/visa

In [ ]:
# Manifest: rebuilt deterministically each session; fails loudly if the locked split changed.
!python -m inspectedge.data.build_manifest

In [ ]:
# Ampere+ GPU (A100 preferred; L4/H100 also fine). bf16 + TF32 need compute capability >= 8.0.
gpu = torch.cuda.get_device_name(0); mem_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
assert torch.cuda.is_bf16_supported() and torch.cuda.get_device_capability(0)[0] >= 8, \
    f'{gpu} lacks bf16: Runtime -> Change runtime type -> A100 (or L4) GPU'
torch.backends.cuda.matmul.allow_tf32 = True; torch.backends.cudnn.allow_tf32 = True
print(gpu, f'{mem_gb:.0f} GB')

In [ ]:
CKPT = f'{DRIVE}/checkpoints'
!ls {CKPT}
# Restore earlier outputs (B3 frozen val thresholds from notebook 02, val metrics) that a runtime reset wiped from local disk.
!mkdir -p reports && rsync -a --update {DRIVE}/reports/ reports/ 2>/dev/null; ls reports

## 1. Validation selection (full val split)

In [ ]:
import os
assert os.path.exists('/content/InspectEdge/data/raw/visa/.complete') and os.getcwd() == '/content/InspectEdge', 'Runtime was reset: re-run every cell from section 0 (setup) first'

RUNS = ['smolvlm500m_lora_r16_lr2e-4_s0']   # add grid runs
for run in RUNS:
    assert os.path.isdir(f'{CKPT}/{run}/final'), f'no adapter at {CKPT}/{run}/final'
    !python scripts/eval_vlm.py --model HuggingFaceTB/SmolVLM-500M-Instruct --adapter {CKPT}/{run}/final --split val --per-cell 0 --tag {run} 2>&1 | tail -22
    assert os.path.exists(f'reports/vlm_eval/{run}_val_metrics.json'), f'val eval of {run} failed: see the log above'
!rsync -aR reports {DRIVE}/

In [ ]:
import glob, pandas as pd
KEYS = ['schema_valid_rate','status_acc_strict','anomaly_balanced_acc','anomaly_recall_flagged','false_reassurance_rate',
        'unsupported_finding_rate_on_normals','defect_type_macro_f1','region_iou>=0.3_rate','latency_p95_s']
rows = {}
for f in sorted(glob.glob(f'{DRIVE}/reports/vlm_eval/*_val_metrics.json')) + sorted(glob.glob('reports/vlm_eval/*_val_metrics.json')):
    m = json.load(open(f))
    if m.get('corrupt'): continue
    rows[os.path.basename(f).replace('_val_metrics.json', '')] = {k: m.get(k) for k in KEYS}   # local (newer) wins
assert rows, 'no val metrics found locally or on Drive: run the cell above first'
pd.DataFrame(rows).T.sort_values('status_acc_strict', ascending=False).round(3)

Selection rule (pre-registered): among runs with schema_valid ≥ 0.995 and unsupported_finding_rate_on_normals ≤ 0.10 on val, pick max status_acc_strict (invalid report = wrong); ties → lower false_reassurance.

In [ ]:
FINAL = 'smolvlm500m_lora_r16_lr2e-4_s0'   # <- set from the table above
prereg = {'final_run': FINAL, 'primary_metric': 'anomaly_recall_flagged @ val-frozen operating point',
          'gates': 'plan 2.3 + FPR-aware robustness gate', 'b3_thresholds': 'reports/b3_detector_report/frozen_thresholds.json'}
os.makedirs('reports', exist_ok=True); json.dump(prereg, open('reports/preregistration.json','w'), indent=2)
!rsync -aR reports {DRIVE}/

## 2. Locked test — run ONCE

In [ ]:
import os
assert os.path.exists('/content/InspectEdge/data/raw/visa/.complete') and os.getcwd() == '/content/InspectEdge', 'Runtime was reset: re-run every cell from section 0 (setup) first'

assert os.path.exists('reports/preregistration.json'), 'run the pre-registration cell above first'
# B3 test reuses thresholds frozen on val (normally by notebook 02); freeze them here if they were never synced to Drive.
if not os.path.exists('reports/b3_detector_report/frozen_thresholds.json'):
    !python scripts/b3_detector_report.py --split val 2>&1 | tail -5
!python scripts/b3_detector_report.py --split test 2>&1 | tail -5
!python scripts/eval_vlm.py --model HuggingFaceTB/SmolVLM-500M-Instruct --split test --per-cell 0 2>&1 | tail -5
!python scripts/eval_vlm.py --model HuggingFaceTB/SmolVLM-500M-Instruct --adapter {CKPT}/{FINAL}/final --split test --per-cell 0 --tag {FINAL} 2>&1 | tail -5
SYSTEMS = {'B3': 'reports/b3_detector_report/test_predictions.jsonl',
           'B2': 'reports/vlm_eval/SmolVLM-500M-Instruct_test_metrics.json',
           'FT': f'reports/vlm_eval/{FINAL}_test_metrics.json'}
failed = [k for k, p in SYSTEMS.items() if not os.path.exists(p)]
assert not failed, f'test run failed for {failed}: see the logs above'
!rsync -aR reports {DRIVE}/
!python scripts/compare_and_gate.py --split test --system B3=reports/b3_detector_report/test_predictions.jsonl --system B2=reports/vlm_eval/SmolVLM-500M-Instruct_test_predictions.jsonl --system FT=reports/vlm_eval/{FINAL}_test_predictions.jsonl
!rsync -aR reports {DRIVE}/

## 3. Robustness suite on held-out test originals (report each family separately)

In [ ]:
import os
assert os.path.exists('/content/InspectEdge/data/raw/visa/.complete') and os.getcwd() == '/content/InspectEdge', 'Runtime was reset: re-run every cell from section 0 (setup) first'

for fam in ['low_light','defocus_blur','motion_blur','sensor_noise','jpeg','glare','combined']:
    !python scripts/eval_vlm.py --model HuggingFaceTB/SmolVLM-500M-Instruct --adapter {CKPT}/{FINAL}/final --split test --per-cell 5 --corrupt {fam} --severity 3 --tag {FINAL} 2>&1 | tail -22
    assert os.path.exists(f'reports/vlm_eval/{FINAL}_test_{fam}3_metrics.json'), f'robustness eval {fam} failed: see the log above'
!rsync -aR reports {DRIVE}/

## 4. Human grounding review
Download `reports/human_review_test.csv`, have two raters fill it blind, adjudicate disagreements, then compute the unsupported-assertion rate (gate ≤ 2%).

In [ ]:
# Persist outputs to Drive (run at the end and after long steps)
!rsync -aR reports data/manifests {DRIVE}/
print('synced to', DRIVE)